In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import asyncio
import json
from pathlib import Path

import dotenv
import pandas as pd
from tqdm.asyncio import tqdm

dotenv.load_dotenv()


from explain.llm._client import LLMClient, LLMConfig

In [3]:
# Load and process the TSV data
data_path = "/Users/emmanuel.noutahi/Code/hooke-explain/data/curation_v1/perturbations/all_smiles_to_synonym.tsv"
df = pd.read_csv(data_path, sep="\t")

df = df.dropna(subset=["synonyms"])
grouped_df = df.groupby("smiles")["synonyms"].apply(list).reset_index()
grouped_df.columns = ["smiles", "synonym_list"]

In [8]:

# Set up LiteLLM client
config = LLMConfig(
    provider="litellm",
    model="gpt-4o",
)

client = LLMClient(config)

2025-08-12 13:11:45.432 | INFO     | explain.llm._client:__init__:577 - Initialized LiteLLM client with model gpt-4o
2025-08-12 13:11:45.432 | INFO     | explain.llm._client:__init__:694 - Initialized unified LLM client with provider: litellm


In [10]:
def create_molecule_name_prompt(molecules_data):
    """Create a prompt for the LLM to select the top 3 most recognizable names for molecules."""
    prompt = """You are a chemistry expert helping to identify the most recognizable and commonly used names for chemical compounds.

For each molecule below, I will provide:
1. SMILES notation (chemical structure)
2. A list of synonyms/names found in various databases

Your task is to:
1. Select the TOP 3 most well-known, commonly used, or scientifically recognized names from the PROVIDED synonym list
2. Prioritize names in this order:
   - Common drug names (if it's a pharmaceutical compound)
   - Well-known scientific names
   - Standard chemical names (IUPAC preferred)
   - Research compound codes (if widely recognized)
3. Provide a brief reason for each selection
4. If you don't recognize the compound or none of the names seem well-known, return null for that molecule
5. IMPORTANT: Only select names that exist in the provided synonym list

Return ONLY a valid JSON array (no markdown, no code blocks) where each object has:
{
  "smiles": "the SMILES string",
  "top_names": ["name1", "name2", "name3"] or null,
  "reasons": ["reason1", "reason2", "reason3"] or null
}

Here are the molecules to analyze:

"""

    for i, mol_data in enumerate(molecules_data, 1):
        prompt += f"\n{i}. SMILES: {mol_data['smiles']}\n"
        prompt += f"   Synonyms: {mol_data['synonyms']}\n"

    prompt += "\nReturn ONLY the JSON array (no markdown formatting):"
    return prompt


def parse_llm_response(response_content):
    """Parse LLM response, handling markdown code blocks and other formatting."""
    try:
        # Remove markdown code blocks if present
        content = response_content.strip()
        if content.startswith("```json"):
            content = content[7:]  # Remove ```json
        elif content.startswith("```"):
            content = content[3:]  # Remove ```

        if content.endswith("```"):
            content = content[:-3]  # Remove closing ```

        content = content.strip()

        # Parse JSON
        return json.loads(content)

    except json.JSONDecodeError as e:
        print(f"JSON parsing failed: {e}")
        print(f"Content to parse: {content[:200]}...")
        return None
    except Exception as e:
        print(f"Unexpected error parsing response: {e}")
        return None


async def process_molecules_batch(client, molecules_batch, semaphore, batch_id, checkpoint_dir):
    """Process a batch of molecules through the LLM and save immediately."""
    batch_file = Path(checkpoint_dir) / f"batch_{batch_id:04d}.json"

    # Check if this batch was already processed
    if batch_file.exists():
        print(f"Batch {batch_id} already exists, skipping...")
        with open(batch_file) as f:
            return json.load(f)

    async with semaphore:
        try:
            # Prepare batch data
            batch_data = []
            for _, row in molecules_batch.iterrows():
                batch_data.append({"smiles": row["smiles"], "synonyms": row["synonym_list"]})

            # Create prompt and call LLM
            prompt = create_molecule_name_prompt(batch_data)
            messages = [{"role": "user", "content": prompt}]
            response = await client.agenerate(messages)

            # Parse response with improved parser
            results = parse_llm_response(response.content)

            if results is None:
                print(f"Failed to parse LLM response for batch {batch_id}, using empty results")
                # Return empty results for this batch
                results = [
                    {"smiles": row["smiles"], "top_names": None, "reasons": None}
                    for _, row in molecules_batch.iterrows()
                ]

            # Save batch result immediately
            with open(batch_file, "w") as f:
                json.dump(results, f, indent=2)
            print(f"Saved batch {batch_id} to {batch_file}")

            return results

        except Exception as e:
            print(f"Batch {batch_id} processing failed: {e}")
            # Create empty results and save them
            results = [
                {"smiles": row["smiles"], "top_names": None, "reasons": None} for _, row in molecules_batch.iterrows()
            ]

            with open(batch_file, "w") as f:
                json.dump(results, f, indent=2)
            print(f"Saved failed batch {batch_id} with empty results")

            return results


def create_batches(df, batch_size=5):
    """Create batches of molecules for processing."""
    batches = []
    for i in range(0, len(df), batch_size):
        batches.append(df.iloc[i : i + batch_size])
    return batches


def load_all_batch_results(checkpoint_dir):
    """Load all completed batch results from individual files."""
    checkpoint_path = Path(checkpoint_dir)
    if not checkpoint_path.exists():
        return []

    all_results = []
    batch_files = sorted(checkpoint_path.glob("batch_*.json"))

    for batch_file in batch_files:
        try:
            with open(batch_file) as f:
                batch_results = json.load(f)
                all_results.extend(batch_results)
        except Exception as e:
            print(f"Error loading {batch_file}: {e}")

    return all_results


async def process_with_checkpoint(
    client,
    df,
    batch_size=10,
    max_batches=None,
    max_concurrent=3,
    checkpoint_dir="../../data/curation_v1/perturbations/cmpds_names/",
):
    """Process molecules with batch-level checkpointing."""

    # Setup
    checkpoint_path = Path(checkpoint_dir)
    checkpoint_path.mkdir(parents=True, exist_ok=True)

    # Load existing results from individual batch files
    existing_results = load_all_batch_results(checkpoint_dir)
    processed_smiles = {r["smiles"] for r in existing_results}

    if existing_results:
        print(
            f"Found {len(existing_results)} existing results from {len(list(checkpoint_path.glob('batch_*.json')))} batch files"
        )

    # Filter out already processed molecules
    remaining_df = df[~df["smiles"].isin(processed_smiles)].reset_index(drop=True)

    if len(remaining_df) == 0:
        print("All molecules already processed!")
        return pd.DataFrame(existing_results)

    # Create batches from remaining molecules
    batches = create_batches(remaining_df, batch_size)
    if max_batches:
        batches = batches[:max_batches]

    print(f"Processing {len(batches)} new batches ({len(remaining_df)} molecules)")

    # Calculate starting batch ID (based on existing batch files)
    existing_batch_files = list(checkpoint_path.glob("batch_*.json"))
    if existing_batch_files:
        start_batch_id = len(existing_batch_files)
    else:
        start_batch_id = 0

    # Process batches with individual saving
    semaphore = asyncio.Semaphore(max_concurrent)
    tasks = [
        process_molecules_batch(client, batch, semaphore, start_batch_id + i, checkpoint_dir)
        for i, batch in enumerate(batches)
    ]

    # Run with progress bar
    print("Processing batches (each batch saved immediately)...")
    batch_results = await tqdm.gather(*tasks, desc="Processing")

    # Load all results (existing + new)
    all_results = load_all_batch_results(checkpoint_dir)

    # Also save consolidated results
    consolidated_file = checkpoint_path / "all_results.json"
    with open(consolidated_file, "w") as f:
        json.dump(all_results, f, indent=2)

    print(f"Processed {len(batch_results)} new batches")
    print(f"Total results: {len(all_results)} molecules")
    print(f"Consolidated results saved to: {consolidated_file}")

    return pd.DataFrame(all_results)


def create_final_dataframe(results_df):
    """Create the final dataframe with separate columns for each top name and reason."""
    final_data = []

    for _, row in results_df.iterrows():
        result_row = {
            "smiles": row["smiles"],
            "top_name_1": None,
            "top_name_2": None,
            "top_name_3": None,
            "reason_1": None,
            "reason_2": None,
            "reason_3": None,
        }

        if row.get("top_names") and isinstance(row["top_names"], list):
            for i, name in enumerate(row["top_names"][:3]):
                result_row[f"top_name_{i + 1}"] = name

        if row.get("reasons") and isinstance(row["reasons"], list):
            for i, reason in enumerate(row["reasons"][:3]):
                result_row[f"reason_{i + 1}"] = reason

        final_data.append(result_row)

    return pd.DataFrame(final_data)


def check_progress(checkpoint_dir="../../data/curation_v1/perturbations/cmpds_names/"):
    """Check current progress."""
    checkpoint_path = Path(checkpoint_dir)
    if not checkpoint_path.exists():
        print("No checkpoint directory found")
        return 0, 0

    # Count batch files
    batch_files = list(checkpoint_path.glob("batch_*.json"))
    print(f"Found {len(batch_files)} completed batches")

    # Load all results
    all_results = load_all_batch_results(checkpoint_dir)
    successful = sum(1 for r in all_results if r.get("top_names"))

    print(f"Total molecules: {len(all_results)}")
    print(f"Successful: {successful}")

    return len(all_results), successful


def clear_results(checkpoint_dir="../../data/curation_v1/perturbations/cmpds_names/"):
    """Clear saved results to start fresh."""
    checkpoint_path = Path(checkpoint_dir)
    if checkpoint_path.exists():
        import shutil

        shutil.rmtree(checkpoint_path)
        print(f"Cleared all results from {checkpoint_path}")
    else:
        print("No results to clear")


print("All functions loaded with per-batch saving!")

All functions loaded with per-batch saving!


In [12]:
results_df = await process_with_checkpoint(
    client,
    grouped_df,
    batch_size=10,
    max_batches=None,  # Set to None for full dataset
    max_concurrent=10,
)
final_df = create_final_dataframe(results_df)

Processing 947 new batches (9463 molecules)
Processing batches (each batch saved immediately)...


Processing:   0%|          | 0/947 [00:00<?, ?it/s]

Processing:   0%|          | 1/947 [00:06<1:36:00,  6.09s/it]

Saved batch 86 to ../../data/curation_v1/perturbations/cmpds_names/batch_0086.json


Processing:   0%|          | 2/947 [00:06<41:56,  2.66s/it]  

...

Saved batch 241 to ../../data/curation_v1/perturbations/cmpds_names/batch_0241.json


Processing: 100%|█████████▉| 946/947 [20:19<00:01,  1.04s/it]

Saved batch 631 to ../../data/curation_v1/perturbations/cmpds_names/batch_0631.json


Processing: 100%|██████████| 947/947 [20:25<00:00,  1.29s/it]

Saved batch 787 to ../../data/curation_v1/perturbations/cmpds_names/batch_0787.json
Processed 947 new batches
Total results: 9452 molecules
Consolidated results saved to: ../../data/curation_v1/perturbations/cmpds_names/all_results.json


In [14]:
# Optional: Save results to file
output_path = "/Users/emmanuel.noutahi/Code/hooke-explain/data/curation_v1/perturbations/molecules_top_names_sample.tsv"
final_df.to_csv(output_path, index=False)

In [15]:
aboutamol = pd.read_parquet("../../data/curation_v1/perturbations/aboutamol.parquet")

In [23]:
from rdkit.Chem import AllChem


def to_inchi_key(smiles):
    try:
        return AllChem.MolToInchiKey(AllChem.MolFromSmiles(smiles))
    except Exception:
        return None


final_df["inchi_key"] = final_df["smiles"].apply(to_inchi_key)

[14:43:03] SMILES Parse Error: unclosed ring for input: 'CC1=NC2=C(N1)C(=O)N1CCOC(C1)C(O)=O'
[14:43:03] Explicit valence for atom # 6 C, 5, is greater than permitted
[14:43:03] SMILES Parse Error: extra close parentheses while parsing: COC1=CC(CNC(=O)NC2=CC=C(Cl)C=C2)N2CCCC2)=CC=C1
[14:43:03] SMILES Parse Error: check for mistakes around position 40:
[14:43:03] CC=C(Cl)C=C2)N2CCCC2)=CC=C1
[14:43:03] ~~~~~~~~~~~~~~~~~~~~^
[14:43:03] SMILES Parse Error: Failed parsing SMILES 'COC1=CC(CNC(=O)NC2=CC=C(Cl)C=C2)N2CCCC2)=CC=C1' for input: 'COC1=CC(CNC(=O)NC2=CC=C(Cl)C=C2)N2CCCC2)=CC=C1'
[14:43:03] SMILES Parse Error: unclosed ring for input: 'COC1=CC=C(N2C(SCCCN3C(=O)C4=CC=CC5=C4C=CC=C5)NN=C2C2=CC=NC=C2)C=C1'
[14:43:04] SMILES Parse Error: extra open parentheses while parsing: [H][C@@]12CCCN1C(=O)[C@H](C(C)C)NC(=O)[C@@H](NC(=O)C1=CC=C(C)C3=C1N=C1C(=C(C)C(=O)C(N)=C1C(=O)N[C@H]1[C@@H](C)OC(=O)[C@H](C(C)C)N(C)C(=O)CN(C)C(=O)[C@]4([H])CCCN4C(=O)[C@H](C(C)C)NC1=O)[C@@H](C)OC(=O)[C@H](C(C)C)N(C)C(=

In [25]:
grouped_df["inchi_key"] = grouped_df["smiles"].apply(to_inchi_key)

In [57]:
final_df_picked = final_df[(final_df.smiles.isin(grouped_df.smiles)) & (~final_df.duplicated(subset="smiles"))]
final_df_picked.to_csv(output_path, index=False)

In [73]:
tmp_grouped_df = grouped_df[~grouped_df.smiles.isin(final_df_picked.smiles)]
tmp_results_df = await process_with_checkpoint(
    client,
    tmp_grouped_df,
    batch_size=5,
    max_batches=None,  # Set to None for full dataset
    max_concurrent=5,
    checkpoint_dir="../../data/curation_v1/perturbations/cmpds_names_v2/",
)
final_df_recomputed = create_final_dataframe(tmp_results_df)

Found 39 existing results from 8 batch files
Processing 1 new batches (4 molecules)
Processing batches (each batch saved immediately)...


Processing:   0%|          | 0/1 [00:00<?, ?it/s]

Processing: 100%|██████████| 1/1 [00:07<00:00,  7.26s/it]

Saved batch 8 to ../../data/curation_v1/perturbations/cmpds_names_v2/batch_0008.json
Processed 1 new batches
Total results: 43 molecules
Consolidated results saved to: ../../data/curation_v1/perturbations/cmpds_names_v2/all_results.json


In [80]:
tmp = final_df_recomputed.copy()


def recheck_smiles(row):
    # this is to fix wrong prompt design -_-
    if row.smiles in grouped_df.smiles:
        return row.smiles  # nothing to do, LLM did not hallunicate output
    # find the most likely smiles in groupe_df
    top_names = (row.top_name_1, row.top_name_2, row.top_name_3)
    if top_names == (None, None, None):
        return None
    # Find the row in grouped_df whose synonym_list has the most overlap with top_names
    overlap_counts = grouped_df["synonym_list"].apply(lambda syn: len([x for x in top_names if x in syn]))
    max_count = overlap_counts.max()
    if max_count > 0 and (overlap_counts == max_count).sum() == 1:
        best_idx = overlap_counts.idxmax()
        return grouped_df.loc[best_idx, "smiles"]
    else:
        print(f"No unique match found for {row.smiles}")
    return None


tmp["smiles"] = tmp.apply(recheck_smiles, axis=1)

No unique match found for [H][C@@]12CCCN1C(=O)[C@H](C(C)C)NC(=O)[C@@H](NC(=O)C1=CC=C(C)C3=C1N=C1C(=C(C)C(=O)C(N)=C1C(=O)N[C@H]1[C@@H](C)OC(=O)[C@H](C(C)C)N(C)C(=O)CN(C)C(=O)[C@]4([H])CCCN4C(=O)[C@H](C(C)C)NC1=O)[C@@H](C)OC(=O)[C@H](C(C)C)N(C)C(=O)CN(C)C2=O
No unique match found for [H][C@@]12CCCN1C(=O)[C@H](C(C)C)NC(=O)[C@@H](NC(=O)C1=CC=C(C)C3=C1N=C1C(=C(C)C(=O)C(N)=C1C(=O)N[C@H]1[C@@H](C)OC(=O)[C@H](C(C)C)N(C)C(=O)CN(C)C(=O)[C@]4([H])CCCN4C(=O)[C@H](C(C)C)NC1=O)O3)[C@@H](C)OC(=O)[C@H](C(C)C)N(C)C(=O)CN(C)C2=O


In [83]:
tmp = tmp.dropna(subset="smiles")
final_df_picked = pd.concat([final_df_picked, tmp])

In [88]:
final_df_picked[final_df.duplicated("smiles")]

/var/folders/rl/wwcfdj4x0pg293bfqszl970r0000gq/T/ipykernel_23298/104208150.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  final_df_picked[final_df.duplicated("smiles")]


,smiles,top_name_1,top_name_2,top_name_3,reason_1,reason_2,reason_3,inchi_key


In [91]:
final_df_picked = grouped_df.merge(final_df_picked.drop(columns="inchi_key"), on="smiles", how="left")

In [92]:
final_df_picked.to_parquet(
    "/Users/emmanuel.noutahi/Code/hooke-explain/data/curation_v1/perturbations/molecules_selected.parquet"
)

In [126]:
filtered_aboutamol = aboutamol[aboutamol.inchi_key.isin(grouped_df.inchi_key) & (aboutamol.less_than_10um is True)]
filtered_aboutamol.to_parquet(
    "/Users/emmanuel.noutahi/Code/hooke-explain/data/curation_v1/perturbations/potentially_active.parquet"
)

In [ ]:
from collections import defaultdict

import numpy as np
import pandas as pd

# Potency direction sets
LOWER_IS_BETTER = {"IC50", "EC50", "Ki", "Kd"}
HIGHER_IS_BETTER = {"pIC50", "pEC50", "pKi", "pKd"}


def _ascending_for_mt(mt: str) -> bool:
    """True if lower values mean more potent."""
    return str(mt) in LOWER_IS_BETTER


def _targets_list(t):
    if pd.isna(t) or t is None:
        return []
    return [x.strip() for x in str(t).replace("|", ";").split(";") if x.strip()]


def _rank_score_row(mt, vmin, vmax):
    """Higher score = more potent (for sorting descending)."""
    if _ascending_for_mt(mt):
        return -float(vmin)  # lower is better
    return float(vmax)  # higher is better for pX


def _diverse_round_robin(ordered_df, target_col="mce_target", k=10):
    """Round-robin selection to ensure target diversity."""
    buckets = defaultdict(list)
    for i, t in enumerate(ordered_df[target_col].tolist()):
        toks = _targets_list(t)
        buckets[toks[0] if toks else "__unknown__"].append(i)
    picks, keys = [], list(buckets.keys())
    while len(picks) < min(k, len(ordered_df)):
        advanced = False
        for bk in keys:
            if buckets[bk]:
                picks.append(buckets[bk].pop(0))
                advanced = True
                if len(picks) >= k:
                    break
        if not advanced:
            break
    if len(picks) < min(k, len(ordered_df)):
        for i in range(len(ordered_df)):
            if i not in picks:
                picks.append(i)
            if len(picks) >= k:
                break
    return ordered_df.iloc[picks].reset_index(drop=True)


def _pick_top_with_priority(groups_df: pd.DataFrame, k: int = 10) -> pd.DataFrame:
    """Priority: human+gene > human(no gene) > other species; then diverse & potent."""
    if groups_df.empty:
        return groups_df

    def _order(df):
        return df.sort_values("rank_score", ascending=False, kind="mergesort").reset_index(drop=True)

    b1 = groups_df[
        (groups_df["bioref_organism"] == "Homo sapiens")
        & groups_df["bioref_gene_name"].notna()
        & (groups_df["bioref_gene_name"] != "")
    ]
    b2 = groups_df[
        (groups_df["bioref_organism"] == "Homo sapiens")
        & ~((groups_df["bioref_gene_name"].notna()) & (groups_df["bioref_gene_name"] != ""))
    ]
    b3 = groups_df[groups_df["bioref_organism"] != "Homo sapiens"]

    out, remain = [], k
    for bucket in (_order(b1), _order(b2), _order(b3)):
        if remain <= 0 or bucket.empty:
            continue
        part = _diverse_round_robin(bucket, k=remain)
        out.append(part)
        remain -= len(part)

    return pd.concat(out, ignore_index=True) if out else groups_df.iloc[0:0]


def _collect_sources(df_sub: pd.DataFrame, mt: str):
    """
    Keep only `source` and `publication_doi`, at most 5 entries.
    Selection based on top potency (lowest for IC50/etc., highest for pIC50/etc.).
    """
    cols = ["bioref_source_name", "bioref_article_doi", "bioref_std_measurement_value"]
    have = [c for c in cols if c in df_sub.columns]
    if not have:
        return []

    asc = _ascending_for_mt(mt)
    df_sorted = df_sub.sort_values("bioref_std_measurement_value", ascending=asc, na_position="last")

    seen, out = set(), []
    for _, r in df_sorted.iterrows():
        item = {
            "source": r.get("bioref_source_name"),
            "publication_doi": r.get("bioref_article_doi"),
        }
        key = (item["source"], item["publication_doi"])
        if key not in seen:
            seen.add(key)
            out.append(item)
        if len(out) >= 5:
            break
    return out


def build_pertubation_bioactivities(row):
    """Return compact JSON with aggregated bioactivities and top sources."""
    # ---- synonyms ----
    syns = [row.get("top_name_1"), row.get("top_name_2"), row.get("top_name_3")]
    syns = [s for s in syns if pd.notna(s)]
    extra = row.get("synonym_list", []) or []
    if isinstance(extra, list | tuple) and len(extra) > 0:
        k = min(2, len(extra))
        syns.extend(np.random.choice(extra, size=k, replace=False).tolist())
    synonym_names = sorted(set(syns))

    # ---- fetch activities ----
    tmp_act = filtered_aboutamol[filtered_aboutamol["inchi_key"] == row["inchi_key"]].copy()

    # drop unused
    tmp_act = tmp_act.drop(
        columns=[
            c
            for c in [
                "bioref_original_measurement_value",
                "general_kce",
                "less_than_10um",
                "mce_vendor_id",
                "bioref_original_measurement_unit",
            ]
            if c in tmp_act.columns
        ],
        errors="ignore",
    )

    # ensure required cols
    for c in [
        "mce_target",
        "bioref_gene_name",
        "bioref_organism",
        "bioref_measurement_type",
        "bioref_std_measurement_value",
        "bioref_std_measurement_unit",
        "bioref_source_name",
        "bioref_article_doi",
    ]:
        if c not in tmp_act.columns:
            tmp_act[c] = pd.NA

    # numeric std only
    tmp_act["bioref_std_measurement_value"] = pd.to_numeric(tmp_act["bioref_std_measurement_value"], errors="coerce")
    tmp_act = tmp_act.dropna(subset=["bioref_std_measurement_value"]).copy()
    if tmp_act.empty:
        return json.dumps(
            {
                "synonym_names": synonym_names,
                "smiles": row["smiles"],
                "inchi_key": row["inchi_key"],
                "bioactivities": {"info": {}, "aggregates": []},
            },
            separators=(",", ":"),
        )

    # ---- hoist mce_* info ----
    info = {}
    for f in [
        "mce_product_name",
        "mce_clinical_phase",
        "mce_research_area",
        "mce_pathway",
        "mce_biological_activity",
        "mce_target",
    ]:
        if f in tmp_act.columns:
            vals = [v for v in tmp_act[f].dropna().unique().tolist() if v not in (None, "", "null")]
            if len(vals) == 1:
                info[f] = vals[0]
            elif len(vals) > 1:
                info[f] = " | ".join(sorted(map(str, vals)))

    # ---- aggregate by (target, org, type, unit, gene) ----
    gb = tmp_act.groupby(
        ["mce_target", "bioref_organism", "bioref_measurement_type", "bioref_std_measurement_unit", "bioref_gene_name"],
        dropna=False,
        as_index=False,
    ).agg(
        count=("bioref_std_measurement_value", "size"),
        min=("bioref_std_measurement_value", "min"),
        median=("bioref_std_measurement_value", "median"),
        max=("bioref_std_measurement_value", "max"),
    )

    gb["rank_score"] = [
        _rank_score_row(mt if pd.notna(mt) else "IC50", vmin, vmax)
        for mt, vmin, vmax in zip(gb["bioref_measurement_type"], gb["min"], gb["max"], strict=False)
    ]

    gb_sel = _pick_top_with_priority(gb)

    # ---- build aggregates ----
    aggregates = []
    for _, r in gb_sel.iterrows():
        mask = (
            (tmp_act["mce_target"] == r["mce_target"])
            & (tmp_act["bioref_organism"] == r["bioref_organism"])
            & (tmp_act["bioref_measurement_type"] == r["bioref_measurement_type"])
            & (tmp_act["bioref_std_measurement_unit"] == r["bioref_std_measurement_unit"])
            & (
                tmp_act["bioref_gene_name"].fillna("")
                == (r["bioref_gene_name"] if pd.notna(r["bioref_gene_name"]) else "")
            )
        )
        srcs = _collect_sources(tmp_act.loc[mask], r["bioref_measurement_type"])

        entry = {
            "gene": r["bioref_gene_name"] if pd.notna(r["bioref_gene_name"]) else None,
            "organism": r["bioref_organism"],
            "type": r["bioref_measurement_type"],
            "unit": r["bioref_std_measurement_unit"],
            "sources": srcs,
        }
        if int(r["count"]) == 1:
            entry["value"] = float(r["min"])
        else:
            entry["min"] = float(r["min"])
            entry["median"] = float(r["median"])
            entry["max"] = float(r["max"])
        aggregates.append(entry)

    return json.dumps(
        {
            "smiles": row["smiles"],
            "inchi_key": row["inchi_key"],
            "bioactivities": {"info": info, "aggregates": aggregates},
        },
        separators=(",", ":"),
    )

In [199]:
final_df_picked["bioactivities"] = final_df_picked.apply(build_pertubation_bioactivities, axis=1)

In [200]:
final_df_picked.to_parquet(
    "/Users/emmanuel.noutahi/Code/hooke-explain/data/curation_v1/perturbations/molecules_selected.parquet"
)

In [4]:
final_df_picked = pd.read_parquet(
    "/Users/emmanuel.noutahi/Code/hooke-explain/data/curation_v1/perturbations/molecules_selected.parquet"
)
final_df_picked.head()

,smiles,synonym_list,inchi_key,top_name_1,top_name_2,top_name_3,reason_1,reason_2,reason_3,bioactivities
0,BrC1=C(Br)C(Br)=C2C(=C1Br)N=NN2,"[4,5,6,7-tetrabromobenzotriazole, TBB benzotri...",OMZYUVOATZSGJY-UHFFFAOYSA-N,"4,5,6,7-tetrabromobenzotriazole",TBB,Casein Kinase II Inhibitor I,"4,5,6,7-tetrabromobenzotriazole is a well-know...",TBB is a common abbreviation used in scientifi...,Casein Kinase II Inhibitor I is a recognized n...,"{""synonym_names"":[""4,5,6,7-tetrabromobenzotria..."
1,BrC1=C(C2=NN=C(SCC(=O)NC(C3=CC=CS3)C3=CC=CC=C3...,"[AKOS001119592, Z97151845]",BJPSWVDDULSZMD-UHFFFAOYSA-N,None,None,None,None,None,None,"{""synonym_names"":[""AKOS001119592"",""Z97151845""]..."
2,BrC1=C(CN2C(=O)NC(C2=O)(C2=CC=CC=C2)C2=CC=CC=C...,"[AKOS007932363, Z14471460]",QEUFPVZFLQCQNK-UHFFFAOYSA-N,None,None,None,None,None,None,"{""synonym_names"":[""AKOS007932363"",""Z14471460""]..."
3,BrC1=C(C[N+]2(CCOCCC3[C@H]4C(C)(C)[C@H](C4)CC3...,"[SCHEMBL9359902, CHEMBL4303628, NCGC00388506-0...",DDHUTBKXLWCZCO-GFUWAVFVSA-N,None,None,None,None,None,None,"{""synonym_names"":[""NCGC00388506-02"",""SCHEMBL93..."
4,BrC1=C(F)C(Br)=CC2=C1CCC(C)N2C=O,"[6-Fluoro-5,7-dibromo-2-methyl-1-formyl-1,2,3,...",ZZLQPWXVZCPUGC-UHFFFAOYSA-N,"6-Fluoro-5,7-dibromo-2-methyl-1-formyl-1,2,3,4...",CE3F4,"5,7-Dibromo-6-fluoro-2-methyl-3,4-dihydroquino...","6-Fluoro-5,7-dibromo-2-methyl-1-formyl-1,2,3,4...",CE3F4 is a common research compound code.,"5,7-Dibromo-6-fluoro-2-methyl-3,4-dihydroquino...","{""synonym_names"":[""5,7-Dibromo-6-fluoro-2-meth..."


In [19]:
config = LLMConfig(
    provider="gemini",
    model="gemini-2.5-flash",
)

client = LLMClient(config)

2025-08-12 22:07:29.156 | INFO     | explain.llm._client:__init__:297 - Initialized Google Gemini client with model gemini-2.5-flash
2025-08-12 22:07:29.156 | INFO     | explain.llm._client:__init__:699 - Initialized unified LLM client with provider: gemini


In [20]:
client.generate([{"role": "user", "content": "Hello, how are you?"}])

LLMResponse(content="Hello! I'm doing well, thank you for asking.\n\nAs an AI, I don't have feelings or personal experiences, but I'm ready and able to assist you.\n\nHow are you doing today?", tool_calls=None, messages=[{'role': 'user', 'content': 'Hello, how are you?'}, {'role': 'assistant', 'content': "Hello! I'm doing well, thank you for asking.\n\nAs an AI, I don't have feelings or personal experiences, but I'm ready and able to assist you.\n\nHow are you doing today?"}])

In [ ]:
# Cell 4: Fast processing functions without type hints
def create_bioactivity_prompt(bioactivity_json, template):
    """Create a prompt for bioactivity summarization using the template."""
    return template.replace("{bioactivity_json}", bioactivity_json)


async def generate_bioactivity_summary(client, bioactivity_data, template, semaphore):
    """Generate a bioactivity summary for a single molecule."""
    async with semaphore:
        try:
            prompt = create_bioactivity_prompt(bioactivity_data, template)
            messages = [{"role": "user", "content": prompt}]
            response = await client.agenerate(messages)
            return response.content.strip()
        except Exception as e:
            print(f"Error generating summary: {e}")
            return f"Error generating summary: {str(e)}"


async def process_molecules_batch(client, molecules_batch, template, semaphore, batch_id, checkpoint_dir, pbar=None):
    """Process a batch of molecules and generate summaries with real-time progress."""
    batch_file = checkpoint_dir / f"summary_batch_{batch_id:04d}.json"

    # Check if this batch was already processed
    if batch_file.exists():
        if pbar:
            pbar.update(len(molecules_batch))
        return []  # Return empty since we're not reloading existing results

    try:
        batch_results = []

        for _idx, row in molecules_batch.iterrows():
            # Generate summary for this molecule's bioactivity data
            summary = await generate_bioactivity_summary(client, row["bioactivities"], template, semaphore)

            # Extract synonym names directly without parsing JSON again
            result = {
                "smiles": row["smiles"],
                "inchi_key": row.get("inchi_key"),
                "bioactivity_summary": summary,
                "original_bioactivities": row["bioactivities"],
            }
            batch_results.append(result)

            # Update progress bar immediately after each molecule
            if pbar:
                pbar.update(1)

        # Save batch result immediately
        with open(batch_file, "w") as f:
            json.dump(batch_results, f, indent=2)

        return batch_results

    except Exception as e:
        print(f"Batch {batch_id} processing failed: {e}")
        if pbar:
            pbar.update(len(molecules_batch))
        return []


def create_batches(df, batch_size=5):
    """Create batches of molecules for processing."""
    batches = []
    for i in range(0, len(df), batch_size):
        batches.append(df.iloc[i : i + batch_size])
    return batches


def count_existing_results(checkpoint_dir):
    """Fast count of existing results without loading all JSON."""
    if not checkpoint_dir.exists():
        return 0

    batch_files = list(checkpoint_dir.glob("summary_batch_*.json"))
    return len(batch_files)


async def process_bioactivity_summaries(
    client,
    df,
    template,
    batch_size=3,
    max_batches=None,
    max_concurrent=3,
    checkpoint_dir="../../data/curation_v1/perturbations/bioactivity_summaries",
):
    """Process molecules with real-time progress updates"""

    checkpoint_path = Path(checkpoint_dir)
    checkpoint_path.mkdir(parents=True, exist_ok=True)

    # Fast check for existing results
    existing_batch_count = count_existing_results(checkpoint_path)
    if existing_batch_count > 0:
        print(f"Found {existing_batch_count} existing batch files")

    # Create all batches upfront
    all_batches = create_batches(df, batch_size)
    if max_batches:
        all_batches = all_batches[:max_batches]

    # Skip already processed batches
    remaining_batches = all_batches[existing_batch_count:]

    if len(remaining_batches) == 0:
        print("All batches already processed!")
        return pd.DataFrame()  # Return empty for now, can load results later if needed

    total_molecules = sum(len(batch) for batch in remaining_batches)
    print(f"Processing {len(remaining_batches)} new batches ({total_molecules} molecules)")

    # Process batches one by one for real-time updates
    semaphore = asyncio.Semaphore(max_concurrent)
    all_results = []

    semaphore = asyncio.Semaphore(max_concurrent)

    # Create all tasks at once
    tasks = [
        process_molecules_batch(client, batch, template, semaphore, existing_batch_count + i, checkpoint_path)
        for i, batch in enumerate(remaining_batches)
    ]

    # Run all tasks concurrently with progress tracking
    all_results = []
    batch_results = await tqdm.gather(*tasks, desc="Processing batches")
    for results in batch_results:
        all_results.extend(results)

    print(f"\nProcessed {len(remaining_batches)} new batches")
    print(f"New results: {len(all_results)} molecules")

    return pd.DataFrame(all_results)


def load_all_results_when_needed(base_glob="../../data/curation_v1/perturbations/bioactivity_summaries*/"):
    """Load all results from multiple subdirectories only when explicitly requested."""
    all_results = []
    subdirs = sorted(Path().glob(base_glob))

    for subdir in subdirs:
        batch_files = sorted(subdir.glob("summary_batch_*.json"))
        print(f"Loading {len(batch_files)} batch files from {subdir}...")
        for batch_file in tqdm(batch_files, desc=f"Loading batches from {subdir}"):
            try:
                with open(batch_file) as f:
                    batch_results = json.load(f)
                    all_results.extend(batch_results)
            except Exception as e:
                print(f"Error loading {batch_file}: {e}")

    return pd.DataFrame(all_results)

In [ ]:
from tqdm.auto import tqdm

# Run the processing (start with a small test batch)
template = Path("../../data/curation_v1/templates/bioassay-to-report.txt").read_text()
for i in tqdm(range(0, 10), desc="Processing batches"):
    if i < 2:
        continue
    results_df = await process_bioactivity_summaries(
        client=client,
        df=final_df_picked.iloc[i * 1000 : (i + 1) * 1000],
        template=template,
        batch_size=5,
        max_batches=None,
        max_concurrent=30,
        checkpoint_dir=f"../../data/curation_v1/perturbations/bioactivity_summaries_{i + 1}",
    )

Processing batches:   0%|          | 0/10 [00:00<?, ?it/s]

Found 200 existing batch files
All batches already processed!
Processing 200 new batches (1000 molecules)


Processing batches:  40%|████      | 4/10 [03:01<04:32, 45.44s/it]


Processed 200 new batches
New results: 1000 molecules
Processing 200 new batches (1000 molecules)


2025-08-12 22:28:06.626 | ERROR    | explain.llm._client:agenerate:448 - Gemini async generation failed: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}


...


Processed 200 new batches
New results: 1000 molecules
Processing 200 new batches (1000 molecules)


Processing batches:  90%|█████████ | 9/10 [19:03<02:41, 161.69s/it]


Processed 200 new batches
New results: 1000 molecules
Processing 94 new batches (466 molecules)


Processing batches: 100%|██████████| 10/10 [20:33<00:00, 123.36s/it]


Processed 94 new batches
New results: 466 molecules


In [31]:
processed_bioactivity_summaries = load_all_results_when_needed()

Loading 200 batch files from bioactivity_summaries_1...


Loading batches from bioactivity_summaries_1: 100%|██████████| 200/200 [00:00<00:00, 7852.66it/s]


Loading 94 batch files from bioactivity_summaries_10...


Loading batches from bioactivity_summaries_10: 100%|██████████| 94/94 [00:00<00:00, 8398.08it/s]


Loading 200 batch files from bioactivity_summaries_2...


...

Loading batches from bioactivity_summaries_7: 100%|██████████| 200/200 [00:00<00:00, 12020.65it/s]


Loading 200 batch files from bioactivity_summaries_8...


Loading batches from bioactivity_summaries_8: 100%|██████████| 200/200 [00:00<00:00, 12011.35it/s]


Loading 200 batch files from bioactivity_summaries_9...


Loading batches from bioactivity_summaries_9: 100%|██████████| 200/200 [00:00<00:00, 11208.27it/s]


In [37]:
final_df_picked.merge(
    processed_bioactivity_summaries.drop(columns=["original_bioactivities", "synonym_names"]),
    on=["smiles", "inchi_key"],
    how="inner",
).to_parquet("/Users/emmanuel.noutahi/Code/hooke-explain/data/curation_v1/perturbations/annotated_molecules.parquet")